# User-friendly functions for computing time-overlaps in Slater Determinant (SD), Configuration State Function (CSF), and Configuration Interaction (CI ) bases

## Table of contents <a name="toc"></a>

1. [Worked out workflow](#1)

   1.1. [Reading the data](#1.1)
  
   1.2. [Basis of unique SDs for two timesteps](#1.2)
   
   1.3. [Matrix of CI coeffiicents in a common SDs basis](#1.3)
   
   1.4. [Computing time-overlap matrix in the common SDs basis and corresponding CSF basis](#1.4)
   
   1.5. [Computing time-overlap matrix in the CI basis](#1.5)   `

2. [High-level API](#2)
     

## A. Learning objectives

- To read the CI information from the CP2K output of TD-DFT calculations
- To compute the time-overlaps in the SD, CSF, and CI bases


## B. Use cases

- Computing Slater-derminant time-overlaps
- Computing CSF time-overlaps
- Computing many-body (TD-DFT, TD-DFTB, CI) time-overlaps


## C. Functions

- `libra_py`
  - `citools`
    - `ci`
      - [`overlap`](#overlap-1)
    - `interfaces`
      - [`ci_amplitudes_mtx`](#ci_amplitudes_mtx-1)
      - [`sd_and_csf_overlaps_singlet`](#sd_and_csf_overlaps_singlet-1)
      - [`unique_confs`](#unique_confs-1)
    - `slatdet`
      - [`make_excitation`](#make_excitation-1)
      - [`make_ref_det`](#make_ref_det-1)
  - `packages`
    - `cp2k`
      - `methods`
        - [`read_cp2k_tddfpt_log_file`](#read_cp2k_tddfpt_log_file-1)
        - [`read_homo_index`](#read_homo_index-1)        

In [1]:
import os
import numpy as np
import scipy.sparse as sp
import libra_py.packages.cp2k.methods as CP2K_methods
import libra_py
import libra_py.citools.slatdet as sd
import libra_py.citools.interfaces as interfaces
import libra_py.citools.ci as ci

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

## 1. Worked out workflow
<a name="1"></a>
[Back to TOC](#toc)

### 1.1. Reading the data
<a name="1.1"></a>
[Back to TOC](#toc)

In this tutorial, we will essentially repeat the previous one, but will do it in a more general way - using helper functions and more general use cases.

Also, we will be using two sets of CI coefficients to mimic a real-time calculation of time-overlaps and NACs.

As before, let's get the HOMO index from one of the files:
<a name="read_homo_index-1"></a>

In [2]:
filename = "step_1200.log"
ks_homo_index = CP2K_methods.read_homo_index(filename)
print('The HOMO index is:',ks_homo_index)

The HOMO index is: 40


Let's read the TD-DFT data (configurations and CI coefficients), but this time - from two files:
<a name="read_cp2k_tddfpt_log_file-1"></a>

> **NOTE:** pay attention to the `tolerance` keyword below - we set it to a value lower than beflore; this will include more configurations in the output; as a result, this will make it more likely to have different sets of most relevant configurations at the two geometries - we'll need it to highlight the procedure for defining a common set of unique configurations

In [3]:
lowest_orbital = ks_homo_index - 19
highest_orbital = ks_homo_index + 20
params_tddft = {'number_of_states': 10, 'tolerance': 0.01, 
                'logfile_name': filename, 'isUKS': False, 
                'lowest_orbital': lowest_orbital, 'highest_orbital': highest_orbital}
data1 = CP2K_methods.read_cp2k_tddfpt_log_file(params_tddft)

params_tddft.update({'logfile_name':"step_1201.log"})
data2 = CP2K_methods.read_cp2k_tddfpt_log_file(params_tddft)

We already know how the data looks like, but let's print it for our convenience (to get a better idea of what they are):

In [4]:
data1[1]

[[[40, 41], [40, 44]],
 [[40, 42], [40, 45]],
 [[40, 43], [40, 44], [40, 45]],
 [[40, 44], [40, 46], [40, 43], [40, 41]],
 [[40, 45], [40, 46], [40, 44], [36, 48]],
 [[39, 41], [39, 42], [40, 46], [37, 41], [40, 47]],
 [[40, 46],
  [40, 45],
  [39, 41],
  [40, 44],
  [40, 47],
  [40, 43],
  [39, 42],
  [40, 42]],
 [[38, 41], [39, 42], [40, 47], [37, 41], [35, 41], [38, 42], [36, 42]],
 [[40, 47],
  [37, 41],
  [38, 42],
  [39, 43],
  [39, 41],
  [35, 41],
  [38, 41],
  [40, 49],
  [37, 42]],
 [[39, 42],
  [38, 41],
  [39, 43],
  [38, 42],
  [37, 41],
  [40, 49],
  [35, 41],
  [37, 42],
  [39, 41],
  [36, 41]]]

In [5]:
data1[2]

[[-0.973537, 0.15636],
 [-0.971833, 0.126044],
 [0.938266, -0.247684, -0.199747],
 [-0.871286, 0.337412, -0.214674, -0.130343],
 [-0.868286, 0.355366, 0.217078, 0.101636],
 [0.798763, 0.313484, -0.28438, 0.278284, 0.207673],
 [-0.802833,
  -0.356074,
  -0.229352,
  -0.20926,
  -0.178639,
  -0.152597,
  -0.124262,
  -0.103589],
 [0.711404, -0.470105, 0.313769, 0.250202, -0.211135, -0.131253, -0.107792],
 [0.730334,
  -0.451852,
  0.297089,
  0.170404,
  -0.15826,
  -0.146985,
  -0.120566,
  0.117857,
  0.108233],
 [0.64084,
  0.427694,
  -0.327131,
  -0.280067,
  -0.258855,
  -0.204102,
  -0.135733,
  -0.118107,
  -0.11355,
  -0.108687]]

In [6]:
data2[1]

[[[40, 41], [40, 44]],
 [[40, 42], [40, 45]],
 [[40, 43], [40, 44], [40, 45]],
 [[40, 44], [40, 46], [40, 43], [40, 41]],
 [[40, 45], [40, 46], [40, 44]],
 [[39, 41], [40, 46], [39, 42], [37, 41], [40, 47], [40, 45], [38, 42]],
 [[40, 46], [39, 41], [40, 45], [40, 47], [39, 42], [40, 44], [40, 43]],
 [[38, 41], [39, 42], [40, 47], [37, 41], [35, 41], [38, 42], [36, 42]],
 [[40, 47],
  [38, 42],
  [37, 41],
  [38, 41],
  [39, 43],
  [40, 49],
  [39, 42],
  [37, 42],
  [39, 41],
  [35, 41]],
 [[39, 42],
  [38, 41],
  [37, 41],
  [40, 47],
  [39, 43],
  [39, 41],
  [35, 41],
  [38, 42],
  [40, 49],
  [40, 48]]]

In [7]:
data2[2]

[[0.97436, -0.159046],
 [0.973193, -0.12362],
 [0.943672, -0.23069, -0.196193],
 [0.874951, -0.3299, 0.203421, 0.132253],
 [-0.862402, 0.368182, 0.225193],
 [-0.714957, 0.449076, -0.305988, 0.26334, -0.171336, 0.168877, -0.106305],
 [-0.72005, -0.385483, -0.337783, -0.227255, -0.214784, -0.187025, -0.129517],
 [-0.695686, 0.478911, -0.332414, 0.272027, 0.196227, 0.114292, 0.111709],
 [-0.670641,
  -0.382395,
  -0.360513,
  0.265912,
  -0.259765,
  -0.168456,
  0.147381,
  0.144419,
  0.117982,
  0.104368],
 [0.617312,
  0.393269,
  0.372281,
  0.249935,
  -0.244134,
  -0.184845,
  -0.171188,
  -0.166201,
  -0.158188,
  -0.103809]]

### 1.2. Basis of unique SDs for two timesteps
<a name="1.2"></a>
[Back to TOC](#toc)

Let's determine the unque set of determinants that are present in the lists of configurations of either timesteps. 

We'll use:
<a name="unique_confs-1"></a>

In [8]:
help(interfaces.unique_confs)

Help on function unique_confs in module libra_py.citools.interfaces:

unique_confs(configs1, configs2, n_excited_states)
    Collect unique configurations from two sets of *excited-state*
    configuration lists using deep value comparison, while preserving
    first-occurrence order.
    
    The expected input hierarchy is::
    
        configsX : list (over excited electronic states)
          └── ex : list (configurations in a given excited state)
                └── sd : list (possibly nested; e.g., Slater determinant)
    
    Only the first `n_excited_states` excited states from each input
    are considered. The ground state is *not* included and is assumed
    to be handled separately.
    
    Uniqueness is defined by the *deep contents* of each configuration
    (`sd`), not by object identity. Internally, configurations are
    converted into immutable tuple representations using `freeze()`
    to allow efficient hashing and comparison.
    
    Parameters
    ----------
  

In [9]:
print(interfaces.unique_confs(data1[1], data2[1], 1)) # this will be out SD basis 

[[40, 41], [40, 44]]


In [10]:
print(interfaces.unique_confs(data1[1], data1[1], 11))
print(interfaces.unique_confs(data2[1], data2[1], 11))
print(interfaces.unique_confs(data1[1], data2[1], 11)) # this will be out SD basis 

[[40, 41], [40, 44], [40, 42], [40, 45], [40, 43], [40, 46], [36, 48], [39, 41], [39, 42], [37, 41], [40, 47], [38, 41], [35, 41], [38, 42], [36, 42], [39, 43], [40, 49], [37, 42], [36, 41]]
[[40, 41], [40, 44], [40, 42], [40, 45], [40, 43], [40, 46], [39, 41], [39, 42], [37, 41], [40, 47], [38, 42], [38, 41], [35, 41], [36, 42], [39, 43], [40, 49], [37, 42], [40, 48]]
[[40, 41], [40, 44], [40, 42], [40, 45], [40, 43], [40, 46], [36, 48], [39, 41], [39, 42], [37, 41], [40, 47], [38, 41], [35, 41], [38, 42], [36, 42], [39, 43], [40, 49], [37, 42], [36, 41], [40, 48]]


### 1.3. Matrix of CI coeffiicents in a common SDs basis
<a name="1.3"></a>
[Back to TOC](#toc)

Using the basis of unique SDs that can be found in either of the timesteps, we will extract the CI coefficients for each geometry (separately) into the corresponding matrices. Since for each timestep we use a common basis of SDs, the sizes of the CI matrices will be the same for each timestep, although the corresponding coefficients may be places in different patters - according to the configurations they are associated with. Above, when printing the configurations for each timestep, we have seen that they have distinct (although strongly overlapping) sets of SD, so we can expect the C1 and C2 matrices below have somewhat different patterns:
<a name="ci_amplitudes_mtx-1"></a>

In [11]:
help(interfaces.ci_amplitudes_mtx)

Help on function ci_amplitudes_mtx in module libra_py.citools.interfaces:

ci_amplitudes_mtx(nstates, common_sd_basis, configs, ci_amplitudes)
    Construct the CI coefficient matrix in a common Slater-determinant basis.
    
    The resulting matrix C has the structure::
    
        C[sd_index, state_index]
    
    where:
      - state_index = 0 corresponds to the ground state
      - state_index >= 1 corresponds to excited states
      - sd_index = 0 corresponds to the reference determinant
      - sd_index >= 1 corresponds to determinants in `common_sd_basis`
    
    Parameters
    ----------
    nstates : int
        Total number of electronic states, including the ground state.
    
    common_sd_basis : list
        List of unique Slater determinants forming the global CI basis,
        excluding the reference determinant. The ordering of this list
        defines the row ordering of the CI matrix.
    
    configs : list of list
        `configs[i]` contains the Slater determ

In [12]:
common_sd_basis = interfaces.unique_confs(data1[1], data2[1], 10)
print(len(common_sd_basis))
C1 = interfaces.ci_amplitudes_mtx(11, common_sd_basis, data1[1], data1[2])
C2 = interfaces.ci_amplitudes_mtx(11, common_sd_basis, data2[1], data2[2])

20


Let's examine the configuations present in state 5 (4-th excited state) and their coefficients. We'll look at then at each geometry.

You can see that at geometry 1, there are 5 dominant configurations, while at geometry 2 there are 7. This is not a problem, since all these coefficients will be placed in the corresponding elements of the (20+1)-dimensional vectors. These vectors correspond to 20 unique excited SDs + 1 reference SD

In [13]:
print(data1[1][5])
print(data1[2][5])

[[39, 41], [39, 42], [40, 46], [37, 41], [40, 47]]
[0.798763, 0.313484, -0.28438, 0.278284, 0.207673]


In [14]:
print(data2[1][5])
print(data2[2][5])

[[39, 41], [40, 46], [39, 42], [37, 41], [40, 47], [40, 45], [38, 42]]
[-0.714957, 0.449076, -0.305988, 0.26334, -0.171336, 0.168877, -0.106305]


In [15]:
print(C1.T[6,:])
print(C2.T[6,:])

[ 0.        0.        0.        0.        0.        0.       -0.28438
  0.        0.798763  0.313484  0.278284  0.207673  0.        0.
  0.        0.        0.        0.        0.        0.        0.      ]
[ 0.        0.        0.        0.        0.168877  0.        0.449076
  0.       -0.714957 -0.305988  0.26334  -0.171336  0.        0.
 -0.106305  0.        0.        0.        0.        0.        0.      ]


In [16]:
C1.shape

(21, 11)

In [17]:
common_sd_basis

[[40, 41],
 [40, 44],
 [40, 42],
 [40, 45],
 [40, 43],
 [40, 46],
 [36, 48],
 [39, 41],
 [39, 42],
 [37, 41],
 [40, 47],
 [38, 41],
 [35, 41],
 [38, 42],
 [36, 42],
 [39, 43],
 [40, 49],
 [37, 42],
 [36, 41],
 [40, 48]]

### 1.4. Computing time-overlap matrix in the common SDs basis and corresponding CSF basis
<a name="1.4"></a>
[Back to TOC](#toc)

In the function used below, we will be using the following helper functions - one, for creating the reference determinant (given the number of electrons included in this determinant and the index of the HOMO orbital);
<a name="make_ref_det-1"></a>

In [18]:
help(sd.make_ref_det)

Help on function make_ref_det in module libra_py.citools.slatdet:

make_ref_det(nelec, homo_indx)
    Construct a reference Slater determinant (closed-shell) in a
    spin–orbital representation.
    
    Spin orbitals are labeled by integers:
      +i  → spin-up orbital i
      -i  → spin-down orbital i
    
    The reference determinant corresponds to a closed-shell occupation
    of orbitals from (ncore + 1) through `homo_indx`, where::
    
        ncore = nelec // 2
    
    Parameters
    ----------
    nelec : int
        Total number of electrons.
    
    homo_indx : int
        Index of the highest occupied molecular orbital (HOMO).
    
    Returns
    -------
    list of int
        Reference Slater determinant represented as a list of occupied
        spin orbitals.
    
    Notes
    -----
    - Assumes a closed-shell electronic structure.
    - The ordering of spin orbitals follows (i, -i) for each spatial
      orbital index i.



In [19]:
gs = sd.make_ref_det(40, 40)
print(gs)

[21, -21, 22, -22, 23, -23, 24, -24, 25, -25, 26, -26, 27, -27, 28, -28, 29, -29, 30, -30, 31, -31, 32, -32, 33, -33, 34, -34, 35, -35, 36, -36, 37, -37, 38, -38, 39, -39, 40, -40]


We use this reference to create the excited determinants. All configurations are constructed as canonically-ordered ones
<a name="make_excitation-1"></a>

In [20]:
help(sd.make_excitation)

Help on function make_excitation in module libra_py.citools.slatdet:

make_excitation(ref_det, occ, vir)
    Generate a single excitation from a reference Slater determinant.
    
    This function replaces one occupied spin orbital (`occ`) in the
    reference determinant with a virtual spin orbital (`vir`).
    
    Parameters
    ----------
    ref_det : list of int
        Reference Slater determinant represented as a list of occupied
        spin orbitals.
    
    occ : int
        Occupied spin orbital to be removed.
    
    vir : int
        Virtual spin orbital to be inserted.
    
    Returns
    -------
    list of int
        New Slater determinant corresponding to the excitation.
    
    Raises
    ------
    ValueError
        If `occ` is not present in `ref_det`.
    
    Notes
    -----
    - The returned determinant is a new list; the reference determinant
      is not modified.
    - No checks are performed for duplicate occupations or Pauli
      violations.
    - 

In [21]:
ex1 = sd.make_excitation(gs, 40, 41)
print(ex1)

ex2 = sd.make_excitation(gs, 40, 42)
print(ex2)

[21, -21, 22, -22, 23, -23, 24, -24, 25, -25, 26, -26, 27, -27, 28, -28, 29, -29, 30, -30, 31, -31, 32, -32, 33, -33, 34, -34, 35, -35, 36, -36, 37, -37, 38, -38, 39, -39, 41, -40]
[21, -21, 22, -22, 23, -23, 24, -24, 25, -25, 26, -26, 27, -27, 28, -28, 29, -29, 30, -30, 31, -31, 32, -32, 33, -33, 34, -34, 35, -35, 36, -36, 37, -37, 38, -38, 39, -39, 42, -40]


Using these functions, we create a basis of raw configurations (that is with the indexing consistent with that of the indexing of MOs in electronic structure calculations). 

We then can create a mapped basis using `interfaces.configs_and_T_matrix_singlet` function as explained in the previous tutorials. 

The returned mapped basis determinants live in the space of idices of orbitals that are encoded in the MO time-overlap matrix. They are used to compute the time-overlaps of the Slater determinants using the `sd.slater_overlap_matrix` function.

Togeter with the mapped determinants, the SD-to-CSF transformation matrix `T` is returned by the `interfaces.configs_and_T_matrix_singlet` function, so it is logical to also compute the time-overlap in the CSF basis.

For the sake of user's simplicity, the described workflow is hidded in the `interfaces.sd_and_csf_overlaps_singlet` function:
<a name="sd_and_csf_overlaps_singlet-1"></a>

In [22]:
help(interfaces.sd_and_csf_overlaps_singlet)

Help on function sd_and_csf_overlaps_singlet in module libra_py.citools.interfaces:

sd_and_csf_overlaps_singlet(st_mo, lowest_orbital, highest_orbital, nelec, homo_indx, common_sd_basis, _active_space=None, S=0, Ms=0, max_unpaired=0)
    Compute Slater-determinant (SD) and configuration-state-function (CSF)
    overlap matrices for singlet excitations using molecular-orbital
    time-overlaps.
    
    This function constructs a reference determinant and a set of singly
    excited determinants defined by `common_sd_basis`, maps them into a
    spin-adapted singlet CSF basis, and computes both SD and CSF overlap
    matrices.
    
    Parameters
    ----------
    st_mo : sparse matrix or array-like, shape (2*norb, 2*norb)
        Molecular-orbital time-overlap matrix in the spin–orbital basis.
        Spin-up and spin-down blocks are assumed to be included explicitly.
    
    lowest_orbital : int
        Lowest spatial orbital index (1-based) included in `st_mo`.
    
    highest_or

In [23]:
st_mo = sp.load_npz('St_ks_1200.npz')

homo_indx = 40
lowest_orbital = homo_indx - 19
highest_orbital = homo_indx + 20
nelec = 40
print(len(common_sd_basis))

csf_ovlp, sd_ovlp = interfaces.sd_and_csf_overlaps_singlet(
    st_mo,
    lowest_orbital,
    highest_orbital,
    nelec,
    homo_indx,
    common_sd_basis)

20


/home/alexvakimov/SOFTWARE/libra/_build/src/libra_py/citools/slatdet.py:328: ComplexWarning: Casting complex values to real discards the imaginary part
  S_AB[i, j] = np.linalg.det(S_a) * np.linalg.det(S_b)


In [24]:
print(csf_ovlp.shape)
print(csf_ovlp)

(21, 21)
[[ 9.93595120e-01+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j]
 [ 0.00000000e+00+0.j  9.93697091e-01+0.j  2.33310299e-03+0.j
   1.51875701e-02+0.j -1.88527839e-03+0.j -9.61330353e-03+0.j
  -3.77797507e-03+0.j  1.12195986e-05+0.j  4.54164021e-04+0.j
   6.94139891e-06+0.j -9.61148634e-04+0.j  2.37330072e-03+0.j
   2.64308006e-03+0.j -2.87717789e-03+0.j  4.03965797e-05+0.j
  -9.73246150e-05+0.j -4.39370974e-06+0.j -3.06237899e-03+0.j
  -1.46901026e-05+0.j -6.36778538e-03+0.j -1.75082573e-03+0.j]
 [ 0.00000000e+00+0.j -2.61578682e-03+0.j  9.93374689e-01+0.j
   7.74395355e-03+0.j  1.77980483e-02+0.j -1.90193893e-02+0

In [25]:
print(sd_ovlp.shape)
print(sd_ovlp)

(41, 41)
[[ 9.93595120e-01  1.52251643e-02  1.52251643e-02 ... -5.83885946e-04
   5.60300220e-04  5.60300220e-04]
 [-1.54574818e-02  9.93460231e-01 -2.36859758e-04 ...  9.08358565e-06
  -1.75954239e-03 -8.71665960e-06]
 [-1.54574818e-02 -2.36859758e-04  9.93460231e-01 ... -6.35870180e-03
  -8.71665960e-06 -1.75954239e-03]
 ...
 [ 3.58688505e-04  5.49629452e-06  6.25376283e-03 ...  9.92291049e-01
   2.02268755e-07 -1.08067459e-05]
 [-6.02602121e-04  1.62176552e-03 -9.23385803e-06 ...  3.54118999e-07
   9.93323325e-01 -3.39814573e-07]
 [-6.02602121e-04 -9.23385803e-06  1.62176552e-03 ... -1.00976113e-05
  -3.39814573e-07  9.93323325e-01]]


### 1.5. Computing time-overlap matrix in the CI basis
<a name="1.5"></a>
[Back to TOC](#toc)

With all the above preparations, we now have everything needed to compute the time-overlaps in the CI basis

In [26]:
st_ci = C1.T @ csf_ovlp @ C2

print(st_ci.shape)
print(np.diag(st_ci))
print(st_ci)

(11, 11)
[ 0.99359512+0.j -0.96733766+0.j -0.95516359+0.j  0.97556387+0.j
 -0.92843132+0.j  0.92254552+0.j -0.89693427+0.j  0.907188  +0.j
 -0.95477936+0.j -0.90594026+0.j  0.85525624+0.j]
[[ 9.93595120e-01+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j]
 [ 0.00000000e+00+0.j -9.67337664e-01+0.j -1.37818299e-02+0.j
  -3.01884260e-02+0.j  5.87405947e-03+0.j  3.19820225e-02+0.j
   3.47280746e-03+0.j -3.33767445e-02+0.j  3.17364275e-03+0.j
  -7.98179108e-05+0.j -2.12083973e-03+0.j]
 [ 0.00000000e+00+0.j  1.36940389e-02+0.j -9.55163589e-01+0.j
  -1.12148986e-02+0.j  1.03968940e-02+0.j -1.06696696e-01+0.j
   2.06285577e-02+0.j -4.48774970e-02+0.j -5.95217600e-04+0.j
  -5.52599517e-04+0.j  7.03237641e-04+0.j]
 [ 0.00000000e+00+0.j  4.43575884e-02+0.j  3.79730483e-02+0.j
   9.75563873e-01+0.j -7.34998573e-04+0.j  1.18481291e-01+0.j


## 2. High-level API
<a name="2"></a>
[Back to TOC](#toc)

Now, for more practical pursposes, it would be nice to hide all of the above machniery in a single function. So, here it is:
<a name="overlap-1"></a>

In [27]:
help(ci.overlap)

Help on function overlap in module libra_py.citools.ci:

overlap(st_mo, data1, data2, params)
    Compute the CI-state overlap matrix between two electronic-structure
    datasets using molecular-orbital time overlaps.
    
    This routine:
      1. Builds a common Slater-determinant basis from excited-state
         configurations of both datasets
      2. Constructs CI coefficient matrices in that common basis
      3. Computes SD and CSF overlap matrices (singlet)
      4. Projects the overlaps into the CI-state representation
    
    Parameters
    ----------
    st_mo : sparse matrix or array-like, shape (2*norb, 2*norb)
        Molecular-orbital time-overlap matrix in the spin–orbital basis.
    
    data1, data2 : tuple or list
        Electronic-structure data containers with the following layout::
    
            dataX[1] : list of list
                State-resolved configuration lists for excited states.
                dataX[1][i] contains configurations for excited stat

In [28]:
params = {"homo_indx":40, "nocc":19, "nvirt":20, "nelec":40, "nstates":11 }
s = ci.overlap(st_mo, data1, data2, params)

print(s.shape)
print(s)

(11, 11)
[[ 9.93595120e-01+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   0.00000000e+00+0.j  0.00000000e+00+0.j]
 [ 0.00000000e+00+0.j -9.67337664e-01+0.j -1.37818299e-02+0.j
  -3.01884260e-02+0.j  5.87405947e-03+0.j  3.19820225e-02+0.j
   3.47280746e-03+0.j -3.33767445e-02+0.j  3.17364275e-03+0.j
  -7.98179108e-05+0.j -2.12083973e-03+0.j]
 [ 0.00000000e+00+0.j  1.36940389e-02+0.j -9.55163589e-01+0.j
  -1.12148986e-02+0.j  1.03968940e-02+0.j -1.06696696e-01+0.j
   2.06285577e-02+0.j -4.48774970e-02+0.j -5.95217600e-04+0.j
  -5.52599517e-04+0.j  7.03237641e-04+0.j]
 [ 0.00000000e+00+0.j  4.43575884e-02+0.j  3.79730483e-02+0.j
   9.75563873e-01+0.j -7.34998573e-04+0.j  1.18481291e-01+0.j
  -3.92760623e-02+0.j -1.85417094e-03+0.j  6.51059679e-04+0.j
   9.18382336e-04+0.j -1.13927890e-03+0.j]
 [ 0.00000000e+00+0.j  1.38552465e-02+0.j -1.02201384e-02+0.j
   2.35056226